In [1]:
import requests
import pandas as pd
import time
import random
import re
import html
from urllib.parse import unquote


# ============================================================
# 0. 설정
# ============================================================

OUTPUT_CSV = r"C:\Users\User\steam_top500_games.csv"

APPID_TXT = r"C:\Users\User\steam_top500_appids.txt"

TOP_N = 500

# 게임 하나당 최종적으로 남길 태그 최대 개수
MAX_TAGS = 3

# 요청 간 딜레이
MIN_DELAY = 0.5
MAX_DELAY = 1.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}


# ============================================================
# 1. Steam Tag → 장르형 카테고리 변환 규칙
# ============================================================

TAG_CATEGORY_MAP = {

    # --------------------------------------------------------
    # ACTION
    # --------------------------------------------------------

    "action": "Action",
    "fps": "Action",
    "shooter": "Action",
    "first-person": "Action",
    "third-person shooter": "Action",
    "tps": "Action",
    "arcade": "Action",
    "hack and slash": "Action",
    "beat 'em up": "Action",
    "fighting": "Action",
    "combat": "Action",
    "melee": "Action",
    "action-adventure": "Action",
    "hero shooter": "Action",
    "looter shooter": "Action",
    "twin stick shooter": "Action",
    "bullet hell": "Action",
    "shoot 'em up": "Action",
    "run and gun": "Action",
    "character action game": "Action",

    # --------------------------------------------------------
    # RPG
    # --------------------------------------------------------

    "rpg": "RPG",
    "role-playing": "RPG",
    "action rpg": "RPG",
    "jrpg": "RPG",
    "japanese rpg": "RPG",
    "turn-based rpg": "RPG",
    "turn-based combat": "RPG",
    "party-based rpg": "RPG",
    "dungeon crawler": "RPG",
    "roguelike": "RPG",
    "roguelite": "RPG",
    "souls-like": "RPG",
    "crpg": "RPG",
    "old school": "RPG",
    "character customization": "RPG",
    "leveling": "RPG",
    "loot": "RPG",
    "class-based": "RPG",
    "choices matter": "RPG",
    "stats": "RPG",

    # --------------------------------------------------------
    # ADVENTURE
    # --------------------------------------------------------

    "adventure": "Adventure",
    "story rich": "Adventure",
    "exploration": "Adventure",
    "walking simulator": "Adventure",
    "point & click": "Adventure",
    "point and click": "Adventure",
    "interactive fiction": "Adventure",
    "visual novel": "Adventure",
    "narrative": "Adventure",
    "choices matter": "Adventure",
    "multiple endings": "Adventure",
    "detective": "Adventure",
    "mystery": "Adventure",
    "horror adventure": "Adventure",

    # --------------------------------------------------------
    # STRATEGY
    # --------------------------------------------------------

    "strategy": "Strategy",
    "real time strategy": "Strategy",
    "rts": "Strategy",
    "turn-based strategy": "Strategy",
    "4x": "Strategy",
    "grand strategy": "Strategy",
    "tactical": "Strategy",
    "tactical rpg": "Strategy",
    "turn-based": "Strategy",
    "management": "Strategy",
    "city builder": "Strategy",
    "base building": "Strategy",
    "tower defense": "Strategy",
    "card game": "Strategy",
    "deckbuilding": "Strategy",
    "deck builder": "Strategy",
    "auto battler": "Strategy",
    "political": "Strategy",
    "war": "Strategy",
    "military": "Strategy",
    "economy": "Strategy",
    "resource management": "Strategy",
    "grand strategy": "Strategy",

    # --------------------------------------------------------
    # SIMULATION
    # --------------------------------------------------------

    "simulation": "Simulation",
    "sim": "Simulation",
    "life sim": "Simulation",
    "farming sim": "Simulation",
    "dating sim": "Simulation",
    "vehicle simulation": "Simulation",
    "flight simulation": "Simulation",
    "space sim": "Simulation",
    "building": "Simulation",
    "sandbox": "Simulation",
    "automation": "Simulation",
    "crafting": "Simulation",
    "colony sim": "Simulation",
    "factory building": "Simulation",
    "immersive sim": "Simulation",
    "physics": "Simulation",
    "realistic": "Simulation",

    # --------------------------------------------------------
    # SPORTS
    # --------------------------------------------------------

    "sports": "Sports",
    "sport": "Sports",
    "football": "Sports",
    "soccer": "Sports",
    "basketball": "Sports",
    "baseball": "Sports",
    "hockey": "Sports",
    "golf": "Sports",
    "tennis": "Sports",
    "wrestling": "Sports",
    "boxing": "Sports",
    "fishing": "Sports",
    "skateboarding": "Sports",
    "cycling": "Sports",
    "motorsport": "Sports",

    # --------------------------------------------------------
    # RACING
    # --------------------------------------------------------

    "racing": "Racing",
    "car racing": "Racing",
    "racing sim": "Racing",
    "kart racing": "Racing",
    "arcade racing": "Racing",
    "driving": "Racing",
    "automobile sim": "Racing",
    "offroad": "Racing",

    # --------------------------------------------------------
    # CASUAL
    # --------------------------------------------------------

    "casual": "Casual",
    "puzzle": "Casual",
    "match 3": "Casual",
    "hidden object": "Casual",
    "relaxing": "Casual",
    "family friendly": "Casual",
    "clicker": "Casual",
    "idle": "Casual",
    "board game": "Casual",

    # --------------------------------------------------------
    # INDIE
    # --------------------------------------------------------

    "indie": "Indie",

    # --------------------------------------------------------
    # HORROR
    # --------------------------------------------------------

    "horror": "Horror",
    "survival horror": "Horror",
    "psychological horror": "Horror",
    "horror": "Horror",
    "gore": "Horror",
    "dark": "Horror",

    # --------------------------------------------------------
    # SURVIVAL
    # --------------------------------------------------------

    "survival": "Survival",
    "survival crafting": "Survival",
    "open world survival craft": "Survival",
    "survival game": "Survival",
    "crafting": "Survival",
    "base building": "Survival",
    "permadeath": "Survival",
    "zombies": "Survival",

    # --------------------------------------------------------
    # OPEN WORLD
    # --------------------------------------------------------

    "open world": "Open World",
    "open-world": "Open World",
    "exploration": "Open World",
    "sandbox": "Open World",

    # --------------------------------------------------------
    # MULTIPLAYER
    # --------------------------------------------------------

    "multiplayer": "Multiplayer",
    "online multiplayer": "Multiplayer",
    "online co-op": "Multiplayer",
    "co-op": "Multiplayer",
    "cooperative": "Multiplayer",
    "coop": "Multiplayer",
    "pvp": "Multiplayer",
    "pve": "Multiplayer",
    "competitive": "Multiplayer",
    "massively multiplayer": "Multiplayer",
    "mmo": "Multiplayer",
    "mmorpg": "Multiplayer",

    # --------------------------------------------------------
    # PLATFORMER
    # --------------------------------------------------------

    "platformer": "Platformer",
    "2d platformer": "Platformer",
    "3d platformer": "Platformer",
    "metroidvania": "Platformer",
    "precision platformer": "Platformer",

    # --------------------------------------------------------
    # PUZZLE
    # --------------------------------------------------------

    "puzzle": "Puzzle",
    "logic": "Puzzle",
    "word game": "Puzzle",
    "programming": "Puzzle",

    # --------------------------------------------------------
    # STEALTH
    # --------------------------------------------------------

    "stealth": "Stealth",
    "ninja": "Stealth",
    "assassin": "Stealth",

    # --------------------------------------------------------
    # VR
    # --------------------------------------------------------

    "vr": "VR",
    "virtual reality": "VR",
    "vr only": "VR",

    # --------------------------------------------------------
    # EARLY ACCESS
    # --------------------------------------------------------

    "early access": "Early Access"
}


# ============================================================
# 2. 장르 우선순위
# ============================================================

GENRE_PRIORITY = [

    "Action",
    "RPG",
    "Adventure",
    "Strategy",
    "Simulation",
    "Sports",
    "Racing",
    "Casual",
    "Horror",
    "Survival",
    "Open World",
    "Multiplayer",
    "Platformer",
    "Puzzle",
    "Stealth",
    "VR",
    "Indie",
    "Early Access"
]


# ============================================================
# 3. 태그 정규화
# ============================================================

def normalize_tag(tag):

    if not tag:
        return ""

    tag = str(tag).lower().strip()

    tag = tag.replace(
        "_",
        " "
    )

    tag = re.sub(
        r"\s+",
        " ",
        tag
    )

    return tag


# ============================================================
# 4. Steam Tag → 축약 카테고리
# ============================================================

def convert_tag_to_category(tag):

    normalized = normalize_tag(
        tag
    )

    if not normalized:
        return None

    # --------------------------------------------------------
    # 정확히 일치
    # --------------------------------------------------------

    if normalized in TAG_CATEGORY_MAP:

        return TAG_CATEGORY_MAP[
            normalized
        ]

    # --------------------------------------------------------
    # 부분 문자열 기반 처리
    # --------------------------------------------------------

    partial_rules = [

        # Action
        ("fps", "Action"),
        ("shooter", "Action"),
        ("shooting", "Action"),
        ("hack and slash", "Action"),
        ("beat em up", "Action"),
        ("fighting", "Action"),

        # RPG
        ("rpg", "RPG"),
        ("role playing", "RPG"),
        ("roguelike", "RPG"),
        ("roguelite", "RPG"),
        ("souls like", "RPG"),
        ("dungeon", "RPG"),

        # Strategy
        ("strategy", "Strategy"),
        ("tactical", "Strategy"),
        ("turn based", "Strategy"),
        ("city builder", "Strategy"),
        ("tower defense", "Strategy"),
        ("deckbuild", "Strategy"),

        # Simulation
        ("simulation", "Simulation"),
        ("simulator", "Simulation"),
        ("farming sim", "Simulation"),
        ("life sim", "Simulation"),
        ("management", "Simulation"),
        ("automation", "Simulation"),
        ("factory", "Simulation"),

        # Sports
        ("sports", "Sports"),
        ("football", "Sports"),
        ("soccer", "Sports"),
        ("basketball", "Sports"),
        ("baseball", "Sports"),
        ("golf", "Sports"),
        ("tennis", "Sports"),

        # Racing
        ("racing", "Racing"),
        ("racer", "Racing"),
        ("driving", "Racing"),
        ("motorsport", "Racing"),

        # Horror
        ("horror", "Horror"),
        ("psychological horror", "Horror"),

        # Survival
        ("survival", "Survival"),
        ("zombie", "Survival"),

        # Platformer
        ("platformer", "Platformer"),
        ("metroidvania", "Platformer"),

        # Puzzle
        ("puzzle", "Puzzle"),

        # Stealth
        ("stealth", "Stealth"),

        # Multiplayer
        ("multiplayer", "Multiplayer"),
        ("co-op", "Multiplayer"),
        ("coop", "Multiplayer"),
        ("pvp", "Multiplayer"),
        ("pve", "Multiplayer"),
        ("mmo", "Multiplayer"),

        # Open World
        ("open world", "Open World"),

        # VR
        ("vr", "VR"),

        # Casual
        ("casual", "Casual"),
        ("clicker", "Casual"),
        ("idle", "Casual")
    ]

    for keyword, category in partial_rules:

        if keyword in normalized:

            return category

    return None


# ============================================================
# 5. Genre와 관련된 Tag만 추출
# ============================================================

def filter_tags_by_genre(
    raw_tags,
    genre_string,
    max_tags=MAX_TAGS
):

    """
    Steam의 수많은 Tag를 장르와 관련된
    소수의 카테고리로 축약합니다.

    예:

    FPS
    Shooter
    PvP
    Competitive
    Military
    Multiplayer

    ↓

    Action, Multiplayer

    """

    if not raw_tags:

        return "Unknown"

    # --------------------------------------------------------
    # 공식 Genre
    # --------------------------------------------------------

    official_genres = set()

    if genre_string:

        for genre in str(
            genre_string
        ).split(","):

            genre = genre.strip()

            if genre:

                official_genres.add(
                    genre
                )

    # --------------------------------------------------------
    # Tag → 카테고리
    # --------------------------------------------------------

    category_count = {}

    for tag in raw_tags:

        category = (
            convert_tag_to_category(
                tag
            )
        )

        if category is None:
            continue

        # ----------------------------------------------------
        # 공식 Genre와 연결되는지 확인
        # ----------------------------------------------------

        # Genre가 Unknown이면 일단 사용
        if (
            not official_genres
            or "Unknown" in official_genres
        ):

            allowed = True

        else:

            allowed = False

            for genre in official_genres:

                # Genre 자체와 같은 경우
                if category.lower() == genre.lower():

                    allowed = True
                    break

                # 일부 Genre와 자연스럽게 연결
                if (
                    genre == "Action"
                    and category in [
                        "Action",
                        "Shooter",
                        "Multiplayer"
                    ]
                ):

                    allowed = True
                    break

                if (
                    genre == "Adventure"
                    and category in [
                        "Adventure",
                        "Open World",
                        "Horror"
                    ]
                ):

                    allowed = True
                    break

                if (
                    genre == "RPG"
                    and category in [
                        "RPG",
                        "Action",
                        "Adventure",
                        "Open World"
                    ]
                ):

                    allowed = True
                    break

                if (
                    genre == "Strategy"
                    and category == "Strategy"
                ):

                    allowed = True
                    break

                if (
                    genre == "Simulation"
                    and category == "Simulation"
                ):

                    allowed = True
                    break

                if (
                    genre == "Sports"
                    and category == "Sports"
                ):

                    allowed = True
                    break

                if (
                    genre == "Racing"
                    and category == "Racing"
                ):

                    allowed = True
                    break

        if not allowed:
            continue

        # ----------------------------------------------------
        # 카테고리 빈도 계산
        # ----------------------------------------------------

        category_count[
            category
        ] = category_count.get(
            category,
            0
        ) + 1

    # --------------------------------------------------------
    # 공식 Genre가 직접 존재하면 우선
    # --------------------------------------------------------

    selected = []

    for genre in GENRE_PRIORITY:

        if genre in category_count:

            selected.append(
                genre
            )

    # --------------------------------------------------------
    # 빈도 높은 순으로 보충
    # --------------------------------------------------------

    remaining = sorted(
        category_count.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for category, count in remaining:

        if category not in selected:

            selected.append(
                category
            )

    # --------------------------------------------------------
    # 최대 개수 제한
    # --------------------------------------------------------

    selected = selected[
        :max_tags
    ]

    if not selected:

        return "Unknown"

    return ", ".join(
        selected
    )


# ============================================================
# 6. 공통 GET 요청
# ============================================================

def get_request(
    url,
    params=None,
    timeout=30,
    retries=3
):

    for attempt in range(
        1,
        retries + 1
    ):

        try:

            response = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=timeout
            )

            if response.status_code == 200:

                return response

            print(
                f"HTTP {response.status_code} "
                f"(시도 {attempt}/{retries})"
            )

        except Exception as e:

            print(
                f"요청 오류: {e} "
                f"(시도 {attempt}/{retries})"
            )

        time.sleep(
            attempt * 2
        )

    return None


# ============================================================
# 7. Steam 현재 동시접속자 TOP 500 AppID
# ============================================================

def get_top_500_appids():

    print("=" * 70)
    print(
        "Steam 현재 동시접속자 TOP 500 AppID 수집"
    )
    print("=" * 70)

    url = (
        "https://api.steampowered.com/"
        "ISteamChartsService/GetMostPlayedGames/v1/"
    )

    response = get_request(
        url
    )

    if response is None:

        raise RuntimeError(
            "Steam Most Played API 요청 실패"
        )

    try:

        data = response.json()

    except Exception as e:

        raise RuntimeError(
            f"JSON 변환 실패: {e}"
        )

    ranks = data.get(
        "response",
        {}
    ).get(
        "ranks",
        []
    )

    if not ranks:

        raise RuntimeError(
            "Steam API에서 게임 순위를 가져오지 못했습니다."
        )

    top_games = []

    for rank, game in enumerate(
        ranks[:TOP_N],
        start=1
    ):

        appid = game.get(
            "appid"
        )

        if not appid:
            continue

        current_players = game.get(
            "concurrent_in_game",
            game.get(
                "concurrent_players",
                None
            )
        )

        top_games.append(
            {
                "rank": rank,
                "appid": int(appid),
                "current_players":
                    current_players
            }
        )

    if not top_games:

        raise RuntimeError(
            "TOP 500 AppID를 가져오지 못했습니다."
        )

    print(
        f"최종 AppID 수: "
        f"{len(top_games)}"
    )

    print(
        "\n상위 20개:"
    )

    for game in top_games[:20]:

        players = game[
            "current_players"
        ]

        if players is not None:

            print(
                f"{game['rank']:>3}위 | "
                f"AppID {game['appid']:<10} | "
                f"동접자 {int(players):,}"
            )

        else:

            print(
                f"{game['rank']:>3}위 | "
                f"AppID {game['appid']}"
            )

    return top_games


# ============================================================
# 8. AppID 목록 저장
# ============================================================

def save_appids_to_txt(
    top_games
):

    with open(
        APPID_TXT,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "# Steam Current Players TOP 500\n"
        )

        f.write(
            "# rank,appid,current_players\n"
        )

        for game in top_games:

            f.write(
                f"{game['rank']},"
                f"{game['appid']},"
                f"{game['current_players']}\n"
            )

    print(
        f"\nAppID 목록 저장 완료:"
    )

    print(
        APPID_TXT
    )


# ============================================================
# 9. Steam AppDetails
# ============================================================

def get_app_details(
    appid
):

    url = (
        "https://store.steampowered.com/"
        "api/appdetails"
    )

    params = {
        "appids": appid,
        "l": "english",
        "cc": "kr"
    }

    response = get_request(
        url,
        params=params
    )

    if response is None:

        return None

    try:

        data = response.json()

    except Exception:

        return None

    app_data = data.get(
        str(appid)
    )

    if not app_data:

        return None

    if not app_data.get(
        "success",
        False
    ):

        return None

    game = app_data.get(
        "data",
        {}
    )

    if game.get(
        "type"
    ) != "game":

        return None

    # --------------------------------------------------------
    # 게임명
    # --------------------------------------------------------

    game_name = game.get(
        "name",
        "Unknown"
    )

    # --------------------------------------------------------
    # 공식 Genre
    # --------------------------------------------------------

    genres = game.get(
        "genres",
        []
    )

    genre_list = []

    for genre_data in genres:

        genre_name = genre_data.get(
            "description"
        )

        if genre_name:

            genre_list.append(
                genre_name
            )

    if genre_list:

        genre = ", ".join(
            genre_list
        )

    else:

        genre = "Unknown"

    return {
        "game_name":
            game_name,

        "genre":
            genre
    }


# ============================================================
# 10. Steam Review API
# ============================================================

def get_review_data(
    appid
):

    url = (
        "https://store.steampowered.com/"
        f"appreviews/{appid}"
    )

    params = {
        "json": 1,
        "language": "all",
        "purchase_type": "all",
        "num_per_page": 0
    }

    response = get_request(
        url,
        params=params
    )

    if response is None:

        return {
            "rating": None
        }

    try:

        data = response.json()

    except Exception:

        return {
            "rating": None
        }

    summary = data.get(
        "query_summary",
        {}
    )

    positive = summary.get(
        "total_positive",
        0
    )

    negative = summary.get(
        "total_negative",
        0
    )

    total = (
        positive +
        negative
    )

    if total > 0:

        rating = round(
            positive /
            total *
            100,
            2
        )

    else:

        rating = None

    return {
        "rating": rating
    }


# ============================================================
# 11. Steam Store에서 원본 Tags 가져오기
# ============================================================

def get_raw_tags(
    appid
):

    url = (
        f"https://store.steampowered.com/app/"
        f"{appid}/"
        "?l=english"
    )

    response = get_request(
        url
    )

    if response is None:

        return []

    page = response.text

    tags = []

    patterns = [

        r'/tags/en/([^"\'?#]+)',

        r'/tags/en/([^"\'?#]+)\?'
    ]

    for pattern in patterns:

        found = re.findall(
            pattern,
            page,
            re.IGNORECASE
        )

        for tag in found:

            tag = unquote(
                tag
            )

            tag = html.unescape(
                tag
            )

            tag = tag.replace(
                "+",
                " "
            )

            tag = tag.strip()

            if not tag:
                continue

            if tag not in tags:

                tags.append(
                    tag
                )

    return tags


# ============================================================
# 12. 게임 하나 수집
# ============================================================

def collect_game(
    game
):

    rank = game[
        "rank"
    ]

    appid = game[
        "appid"
    ]

    current_players = game[
        "current_players"
    ]

    print("\n" + "-" * 70)

    print(
        f"[{rank}/{TOP_N}] "
        f"AppID = {appid}"
    )

    # --------------------------------------------------------
    # 동접자는 순위 확인용
    # --------------------------------------------------------

    if current_players is not None:

        print(
            f"동접자 = "
            f"{int(current_players):,}"
        )

    # ========================================================
    # AppDetails
    # ========================================================

    details = get_app_details(
        appid
    )

    if details is None:

        print(
            "→ AppDetails 없음 / 제외"
        )

        return None

    print(
        f"게임명 = "
        f"{details['game_name']}"
    )

    print(
        f"공식 장르 = "
        f"{details['genre']}"
    )

    time.sleep(
        random.uniform(
            MIN_DELAY,
            MAX_DELAY
        )
    )

    # ========================================================
    # Review
    # ========================================================

    review = get_review_data(
        appid
    )

    print(
        f"평점 = "
        f"{review['rating']}"
    )

    time.sleep(
        random.uniform(
            MIN_DELAY,
            MAX_DELAY
        )
    )

    # ========================================================
    # Raw Tags
    # ========================================================

    raw_tags = get_raw_tags(
        appid
    )

    print(
        f"원본 Tag = "
        f"{len(raw_tags)}개"
    )

    # ========================================================
    # 장르와 관련된 Tag만 추출
    # ========================================================

    filtered_tags = filter_tags_by_genre(
        raw_tags=raw_tags,
        genre_string=details["genre"],
        max_tags=MAX_TAGS
    )

    print(
        f"정제된 Tag = "
        f"{filtered_tags}"
    )

    # ========================================================
    # 최종 반환
    # ========================================================

    return {

        "rank":
            rank,

        "appid":
            appid,

        "game_name":
            details[
                "game_name"
            ],

        "rating":
            review[
                "rating"
            ],

        "genre":
            details[
                "genre"
            ],

        "tags":
            filtered_tags
    }


# ============================================================
# 13. CSV 저장
# ============================================================

def save_csv(
    results
):

    if not results:

        return

    df = pd.DataFrame(
        results
    )

    # --------------------------------------------------------
    # AppID 중복 제거
    # --------------------------------------------------------

    df = (
        df
        .drop_duplicates(
            subset=[
                "appid"
            ]
        )
        .sort_values(
            "rank"
        )
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # 컬럼 순서
    # --------------------------------------------------------

    columns = [

        "rank",

        "appid",

        "game_name",

        "rating",

        "genre",

        "tags"
    ]

    df = df[
        columns
    ]

    # --------------------------------------------------------
    # CSV 저장
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nCSV 저장 완료:"
    )

    print(
        OUTPUT_CSV
    )


# ============================================================
# 14. 메인
# ============================================================

def main():

    print("\n")

    print("=" * 70)

    print(
        "STEAM CURRENT PLAYER TOP 500"
    )

    print(
        "장르 기반 Tag 정제 버전"
    )

    print("=" * 70)

    # ========================================================
    # STEP 1
    # 현재 동접자 TOP 500
    # ========================================================

    top_games = (
        get_top_500_appids()
    )

    if not top_games:

        print(
            "TOP 500 AppID를 가져오지 못했습니다."
        )

        return

    # ========================================================
    # STEP 2
    # AppID 저장
    # ========================================================

    save_appids_to_txt(
        top_games
    )

    # ========================================================
    # STEP 3
    # 게임 데이터 수집
    # ========================================================

    results = []

    total = len(
        top_games
    )

    for index, game in enumerate(
        top_games,
        start=1
    ):

        try:

            result = collect_game(
                game
            )

            if result is not None:

                results.append(
                    result
                )

        except Exception as e:

            print(
                f"오류 발생: {e}"
            )

        # ----------------------------------------------------
        # 요청 간 딜레이
        # ----------------------------------------------------

        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )

        # ----------------------------------------------------
        # 20개마다 중간 저장
        # ----------------------------------------------------

        if index % 20 == 0:

            save_csv(
                results
            )

            print(
                f"\n중간 저장: "
                f"{index}/{total}"
            )

    # ========================================================
    # STEP 4
    # 최종 저장
    # ========================================================

    save_csv(
        results
    )

    # ========================================================
    # STEP 5
    # 결과
    # ========================================================

    print("\n")

    print("=" * 70)

    print(
        "전체 수집 완료"
    )

    print("=" * 70)

    print(
        f"목표 게임 수: "
        f"{TOP_N}"
    )

    print(
        f"수집 성공: "
        f"{len(results)}"
    )

    print(
        f"\nAppID 목록:"
    )

    print(
        APPID_TXT
    )

    print(
        f"\n최종 CSV:"
    )

    print(
        OUTPUT_CSV
    )

    # ========================================================
    # 상위 20개 확인
    # ========================================================

    if results:

        df = pd.DataFrame(
            results
        )

        print(
            "\n상위 20개:"
        )

        print(
            df.head(
                20
            ).to_string(
                index=False
            )
        )


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":

    main()



STEAM CURRENT PLAYER TOP 500
장르 기반 Tag 정제 버전
Steam 현재 동시접속자 TOP 500 AppID 수집
최종 AppID 수: 100

상위 20개:
  1위 | AppID 730
  2위 | AppID 570
  3위 | AppID 578080
  4위 | AppID 431960
  5위 | AppID 1172470
  6위 | AppID 1623730
  7위 | AppID 3527290
  8위 | AppID 553850
  9위 | AppID 2767030
 10위 | AppID 250900
 11위 | AppID 271590
 12위 | AppID 3405690
 13위 | AppID 2868840
 14위 | AppID 2676230
 15위 | AppID 359550
 16위 | AppID 2357570
 17위 | AppID 381210
 18위 | AppID 236390
 19위 | AppID 108600
 20위 | AppID 322170

AppID 목록 저장 완료:
C:\Users\User\steam_top500_appids.txt

----------------------------------------------------------------------
[1/500] AppID = 730
게임명 = Counter-Strike 2
공식 장르 = Action, Free To Play
평점 = 85.95
원본 Tag = 20개
정제된 Tag = Action, Multiplayer

----------------------------------------------------------------------
[2/500] AppID = 570
게임명 = Dota 2
공식 장르 = Action, Strategy, Free To Play
평점 = 80.57
원본 Tag = 20개
정제된 Tag = Strategy, Multiplayer

----------------------------------------